<a href="https://colab.research.google.com/github/reynaudnangue28/test/blob/main/CO2_Emission_Project_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Classification using regression models
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from skopt import BayesSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from sklearn.ensemble import RandomForestClassifier

# Google Colab to access and mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ---  load a mapping csv---
try:
    mapping_df = pd.read_csv('/content/drive/MyDrive/Data/EEA2024CO2Data/eea_column_mapping_with_nulls.csv')
    cols_to_keep = mapping_df[mapping_df['keep'] == True]
    rename_map = dict(zip(cols_to_keep['column_name_original'], cols_to_keep['column_name_new']))
except FileNotFoundError:
    print("Error : file is missing.")
    exit()

# ---  load the EEA file  ---

df1 = pd.read_csv(
    "/content/drive/MyDrive/Data/EEA2024CO2Data/data.csv",
    nrows=7000000,
    low_memory=False
)

#Reading Between Raws 7 million and 10.8 million

df2 = pd.read_csv(
    "/content/drive/MyDrive/Data/EEA2024CO2Data/data.csv",
    skiprows=range(1, 7_000_001),  # skip header + first 7M rows
    low_memory=False
)

# ---  Filter and rename ---
# choose only the columns with 'keep=True'
df1 = df1[rename_map.keys()]
# Rename the columns
df1 = df1.rename(columns=rename_map)

# choose only the columns with 'keep=True'
df2 = df2[rename_map.keys()]
# Rename the columns
df2 = df2.rename(columns=rename_map)

#Concat of df1 and df2
df = pd.concat([df1, df2], axis=0, ignore_index=True)

# ---  end ---
#df1 = df1.dropna(subset=['CO2_Emissions_g_km', 'Mass_Min_kg'])
#print(df.info())

df['fuel_type'] = df['fuel_type'].replace({'ES': "Petrol",
                                   'diesel': "Diesel",
                                   'petrol': "Petrol",
                                   'petrol/electric': "Petrol-Electric Hybrid ",
                                   'diesel/electric': "Diesel-Electric Hybrid",
                                   'lpg': "Liquefied Petroleum Gas",
                                   'ng': "Natural Gas",
                                   'electric': "100% Electric",
                                    'e85' : "Ethanol 85%",
                                    'hydrogen' : "Fuel Cell Electric Vehicles"})
#print(df["Fuel_Type"].unique())

#  Basic Preprocessing

#numeric features
features = [
    "mass_kg",
    "wltp_mass_kg",
    "engine_cc",
    "power_kw",
    "electric_consumption",
    "fuel_type"
]
target = 'co2_wltp'

# Drop rows with missing values in these specific columns
df = df.dropna(subset=features + [target])
X = pd.get_dummies(df[features], columns=['fuel_type'], drop_first=True)
y = df[target]

# Create the classes
def categorize_co2(value):
    if value <= 50: return 0 # Low
    elif value <= 150: return 1 # Medium
    else: return 2 # High

y_class = y.apply(categorize_co2)

# New Split for Classification
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_class, test_size=0.2, random_state=42, stratify=y_class
)

#  Select hyperparameters
param_grid_rf = {
    'max_features': ['sqrt', 'log2', None],
    'min_samples_split': list(range(2, 31))
}

# Initialize Classifiers with parallel processing
grid_clf = GridSearchCV(RandomForestClassifier(n_estimators=100), param_grid_rf, cv=5, scoring='accuracy')
ram_clf = RandomizedSearchCV(RandomForestClassifier(n_estimators=100), param_grid_rf, cv=5, scoring='accuracy')
bayes_clf = BayesSearchCV(RandomForestClassifier(n_estimators=100), param_grid_rf, cv=5, scoring='accuracy')

print("Training Classifiers on 10M+ rows...")
grid_clf.fit(X_train_c, y_train_c)
ram_clf.fit(X_train_c, y_train_c)
bayes_clf.fit(X_train_c, y_train_c)

#  Evaluation Function for Classification
def evaluate_classification(model, X_test, y_test, name):
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    print(f"--- {name} Classification Results ---")
    print(f"Accuracy: {acc:.4f}")
    print(classification_report(y_test, preds, target_names=['Low', 'Medium', 'High']))

    # Confusion Matrix
    plt.figure(figsize=(6,4))
    sns.heatmap(confusion_matrix(y_test, preds), annot=True, fmt='d', cmap='Blues')
    plt.title(f"Confusion Matrix - {name}")
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()

evaluate_classification(grid_clf, X_test_c, y_test_c, "Random Forest with GridSearchCV")
evaluate_classification(ram_clf, X_test_c, y_test_c, "Random Forest with RandomSearchCV")
evaluate_classification(bayes_clf, X_test_c, y_test_c, "Random Forest with BayesSearchCV")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Training Classifiers on 10M+ rows...


In [3]:
!pip install opencv-python
!pip install scikit-optimize


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 3.3 MB/s eta 0:00:00


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
